<a href="https://colab.research.google.com/github/mibucko/ml-product-categories/blob/main/product_categories_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is part of a machine learning project in which we are developing a model to predict the product category based on the input data. The documents are available on GitHub at the following link:
https://github.com/mibucko/ml-product-categories

1. Raw data gathering

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/mibucko/ml-product-categories/main/products.csv"
df = pd.read_csv(url)

1.1. Column names standardization

In [2]:
df.columns = (df.columns.str.strip().str.lower().str.replace(r"[\s_]+", "_", regex=True).str.strip("_"))
print(df.columns)

Index(['product_id', 'product_title', 'merchant_id', 'category_label',
       'product_code', 'number_of_views', 'merchant_rating', 'listing_date'],
      dtype='object')


1.2. Brief view on data

In [3]:
print("Number of rows:", len(df))
print("Sample rows:")
print(df.sample(5))

Number of rows: 35311
Sample rows:
       product_id                                      product_title  \
33391       45332   liebherr t1504 fridge freezer 117 litre in white   
30203       41827                                    bosch kgv39vi31   
32433       44223  gorenje retro special edition obrb153bl tall f...   
4651        10623                lg ultra hdr freeview play 4k tv 43   
34664       46685                            neff k hlger t k 246 a3   

       merchant_id   category_label product_code  number_of_views  \
33391          132          Fridges   BU-9162-XV           1852.0   
30203          119  Fridge Freezers   VY-1469-YE           3856.0   
32433            7          Fridges   NH-5450-IM              9.0   
4651            72              TVs   NK-7178-JM           1008.0   
34664          300          Fridges   BU-5657-TP            436.0   

       merchant_rating listing_date  
33391              3.9   12/12/2022  
30203              3.5    3/17/2023  
324

1.3. Dropping columns that definitely cannot influence product category: product_id, merchant_id, number_of_views, merchant_rating and listing_date.

In [4]:
df = df.drop(columns=["product_id", "merchant_id", "number_of_views",
    "merchant_rating", "listing_date"])

2. Exploratory data analysis – EDA

2.1. What categories are present in dataset?

In [5]:
print("Number of categories:", df["category_label"].nunique())
print(df["category_label"].value_counts())

Number of categories: 13
category_label
Fridge Freezers     5495
Washing Machines    4036
Mobile Phones       4020
CPUs                3771
TVs                 3564
Fridges             3457
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
fridge               123
CPU                   84
Mobile Phone          55
Name: count, dtype: int64


2.2. Is the product code related to the product category?

We choose two categories as an example. By visual inspection, we can see that there is no apparent correlation between the product code and the product category. Therefore, we also drop this column from the dataset.

In [6]:
mobile_phones = df[df["category_label"] == "Mobile Phones"]
cameras = df[df["category_label"] == "Digital Cameras"]
print('mobile phones')
print(mobile_phones[["product_code", "category_label"]].sample(20))
print('cameras')
print(cameras[["product_code", "category_label"]].sample(20))

mobile phones
     product_code category_label
3885   TK-2274-HA  Mobile Phones
2625   ZU-9909-JN  Mobile Phones
588    WV-0614-AF  Mobile Phones
3592   UB-6006-EH  Mobile Phones
2646   WN-5074-BS  Mobile Phones
1711   GQ-7030-LU  Mobile Phones
1148   AX-5866-LK  Mobile Phones
238    LS-0505-NO  Mobile Phones
2464   TY-6670-NC  Mobile Phones
1344   OL-3034-RS  Mobile Phones
3324   MU-4071-OO  Mobile Phones
1965   HH-9688-OC  Mobile Phones
2244   PE-5689-OL  Mobile Phones
284    FI-9926-MO  Mobile Phones
2545   ZL-5178-QC  Mobile Phones
1633   UR-9010-AB  Mobile Phones
1925   AQ-5995-AI  Mobile Phones
1395   JL-4726-RG  Mobile Phones
1450   MO-6759-UN  Mobile Phones
21     FD-9547-UE  Mobile Phones
cameras
      product_code   category_label
13728   DN-1307-HV  Digital Cameras
12223   LI-5572-AT  Digital Cameras
13650   CC-7975-AE  Digital Cameras
14097   IU-5798-HV  Digital Cameras
13555   YW-8765-SH  Digital Cameras
13965   RB-5218-HD  Digital Cameras
12883   KT-8008-OI  Digital Camer

In [7]:
df = df.drop(columns=["product_code"])
print('Remaining columns:')
print(df.columns)

Remaining columns:
Index(['product_title', 'category_label'], dtype='object')


3. Data processing

3.1 Merging similar categories

In [8]:
df["category_label"] = df["category_label"].replace({
    "Mobile Phone": "Mobile Phones",
    "CPU": "CPUs",
    "fridge": "Fridges"
})
print(df["category_label"].value_counts())

category_label
Fridge Freezers     5495
Mobile Phones       4075
Washing Machines    4036
CPUs                3855
Fridges             3580
TVs                 3564
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
Name: count, dtype: int64


3.2. Handling Missing Values - isna

There are 171 rows without a product title, so we drop these rows. There are also 43 rows without a product category. We remove these rows from the original dataset and store them separately in a dataset called "only_title", so that we can potentially use them later to test our model.

In [9]:
print("product_title:", df["product_title"].isna().sum())
print("category_label:", df["category_label"].isna().sum())
only_title = df[df["category_label"].isna()]

product_title: 172
category_label: 44


3.2. Handling Missing Values - dropna

We drop rows with missing values from the original dataset.

In [10]:
df = df.dropna(subset=["product_title", "category_label"])

4. Feature engineering

4.1 Brief look at product titles

In [11]:
sample = df.sample(10).sort_values("category_label")
print(sample[["category_label", "product_title"]].to_string(index=False))

  category_label                                                        product_title
 Digital Cameras                             canon eos 200d schwarz ef s 18 55 is stm
     Dishwashers                          kenwood kdw60w15 full size dishwasher white
        Freezers                        beko bz77f integrated tall frost free freezer
 Fridge Freezers         hotpoint h8a1ew 228 111 litre low frost fridge freezer white
 Fridge Freezers   liebherr icun3324 178cm integrated 70/30 frost free fridge freezer
         Fridges     siemens iq 300 ki82lvs30g integrated upright fridge with ice box
         Fridges               bosch kur15a50gb built in under larder fridge in white
   Mobile Phones vkworld stone v3s 32mb 32mb anti low temperature daily waterproof sh
             TVs                                        lg 49uk6300 49 ultra hd 4k tv
Washing Machines   neff w7460x4gb freestanding 9kg 1400rpm washing machine in white a


4.2. Creating new Columns

We create five new features from the product title:

- word_count – number of words in the product title
- char_count – number of characters in the product title
- special_count – number of special characters in the product title
- words_with_digits – number of words containing digits in the title
- max_word_length – length of the longest word in the title

In [12]:
df["word_count"] = df["product_title"].str.split().str.len()
df["char_count"] = df["product_title"].str.replace(" ", "").str.len()
df["special_count"] = df["product_title"].str.count(r"[^A-Za-z0-9\s]")
df["words_with_digits"] = df["product_title"].str.split().apply(
    lambda words: sum(any(char.isdigit() for char in word) for word in words))
df["max_word_length"] = df["product_title"].str.split().apply(
    lambda words: max(len(word) for word in words))

5. Data Preparation for Algorithm Training

5.1 Firstly we define seven experimental data sets:

In [13]:
y = df["category_label"]

feature_sets = {
    "baseline": ["product_title"],
    "word_count": ["product_title", "word_count"],
    "char_count": ["product_title", "char_count"],
    "special_count": ["product_title", "special_count"],
    "words_with_digits": ["product_title", "words_with_digits"],
    "max_word_length": ["product_title", "max_word_length"],
    "all_features": [
        "product_title",
        "word_count",
        "char_count",
        "special_count",
        "words_with_digits",
        "max_word_length"
    ]
}

5.2 Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

5.3. TF-IDF Vectorization: We choose the bigram option for the first tests, but later we also use unigram and trigram.

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Bigram

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True
)

X_train_text = tfidf.fit_transform(
    df.loc[train_idx, "product_title"]
)

X_test_text = tfidf.transform(
    df.loc[test_idx, "product_title"]
)

# Unigram

tfidf_unigram = TfidfVectorizer()

X_train_unigram = tfidf_unigram.fit_transform(
    df.loc[train_idx, "product_title"]
)

X_test_unigram = tfidf_unigram.transform(
    df.loc[test_idx, "product_title"]
)


# Trigram

tfidf_trigram = TfidfVectorizer(
    ngram_range=(1, 3),
    lowercase=True
)

X_train_trigram = tfidf_trigram.fit_transform(
    df.loc[train_idx, "product_title"]
)

X_test_trigram = tfidf_trigram.transform(
    df.loc[test_idx, "product_title"]
)

5.4 Scaling Numerical Features

In [16]:
from sklearn.preprocessing import MinMaxScaler

numeric_features = [
    "word_count",
    "char_count",
    "special_count",
    "words_with_digits",
    "max_word_length"
]

scaler = MinMaxScaler()

X_train_num = scaler.fit_transform(
    df.loc[train_idx, numeric_features]
)

X_test_num = scaler.transform(
    df.loc[test_idx, numeric_features]
)

5.5. Defining a Function for Feature Selection

We will call this function each time we prepare the features for algorithm training.

In [17]:
def get_feature_sets(features):

    if len(features) == 0:
        X_train_final = X_train_text
        X_test_final = X_test_text

    else:
        indices = [numeric_features.index(f) for f in features]

        X_train_final = hstack([
            X_train_text,
            X_train_num[:, indices]
        ])

        X_test_final = hstack([
            X_test_text,
            X_test_num[:, indices]
        ])

    return X_train_final, X_test_final

5.6. Experimental Feature Sets

In [18]:
feature_sets = {
    "Baseline": [],
    "Word count": ["word_count"],
    "Char count": ["char_count"],
    "Special count": ["special_count"],
    "Words with digits": ["words_with_digits"],
    "Max word length": ["max_word_length"],
    "All features": numeric_features
}

5.7. Defining a Function for Training and Evaluation

We will call this function each time we train and evaluate a new model.

In [19]:
def train_and_evaluate(model, model_name):

    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)

    print(f"\n{'='*60}")
    print(f"{model_name} - {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=model.classes_,
        columns=model.classes_
    ))

6. Training and Testing of the Algorithms

We have seven experimental datasets. We will perform training and testing using five different algorithms. Each Colab cell is dedicated to one algorithm and includes all seven experimental datasets.

6.1 Logistic Regression

Please find the metric results below. To select the optimal dataset, we focus our attention on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the **Fridges** category. The **Word count** dataset slightly outperforms the others (F1-score = 0.91), which is why we propose using it for further experimentation with this algorithm.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from scipy.sparse import hstack
from sklearn.metrics import confusion_matrix

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = LogisticRegression(max_iter=1000)
    train_and_evaluate(model, "Logistic Regression")


Logistic Regression - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.93      0.96      0.95       681
        Freezers       0.99      0.93      0.96       440
 Fridge Freezers       0.96      0.94      0.95      1094
         Fridges       0.89      0.92      0.90       712
      Microwaves       0.99      0.95      0.97       466
   Mobile Phones       0.97      0.99      0.98       812
             TVs       0.97      0.99      0.98       708
Washing Machines       0.95      0.94      0.94       803

        accuracy                           0.96      7020
       macro avg       0.96      0.96      0.96      7020
    weighted avg       0.96      0.96      0.96      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               761                0            0         0   
Dig

6.2 Naive Bayes

Please find the metric results below. To select the optimal dataset, we focus our attention on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the Freezers category. The Baseline dataset and dataset where special characters were counted, slightly outperform the others (F1-score = 0.80), which is why we propose using these two datasets for further experimentation with this algorithm.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = MultinomialNB()
    train_and_evaluate(model, "Naive Bayes")


Naive Bayes - Baseline
                  precision    recall  f1-score   support

            CPUs       0.99      1.00      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.99      0.94      0.96       681
        Freezers       1.00      0.67      0.80       440
 Fridge Freezers       0.80      0.99      0.88      1094
         Fridges       0.94      0.87      0.90       712
      Microwaves       0.99      0.96      0.98       466
   Mobile Phones       0.99      0.99      0.99       812
             TVs       0.98      0.99      0.99       708
Washing Machines       0.98      0.96      0.97       803

        accuracy                           0.95      7020
       macro avg       0.97      0.94      0.95      7020
    weighted avg       0.96      0.95      0.95      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               765                0            0         0   
Digital Cam

6.3. Decision Tree

Please find the metric results below. To select the optimal dataset, we focus on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the Fridges category. Three datasets slightly outperform the others (F1-score = 0.87 vs. 0.86). Therefore, we propose using these three datasets for further experimentation with this algorithm: the dataset with character count, the dataset with maximum word length, and the dataset with all features.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = DecisionTreeClassifier()
    train_and_evaluate(model, "Decision Tree")


Decision Tree - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      1.00       766
 Digital Cameras       0.99      0.94      0.97       538
     Dishwashers       0.94      0.90      0.92       681
        Freezers       0.89      0.91      0.90       440
 Fridge Freezers       0.93      0.89      0.91      1094
         Fridges       0.86      0.87      0.86       712
      Microwaves       0.88      0.93      0.90       466
   Mobile Phones       0.93      0.98      0.95       812
             TVs       0.93      0.96      0.94       708
Washing Machines       0.93      0.92      0.93       803

        accuracy                           0.93      7020
       macro avg       0.93      0.93      0.93      7020
    weighted avg       0.93      0.93      0.93      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               761                0            0         0   
Digital C

6.4. Random Forest

Please find the metric results below. To select the optimal dataset, we focus our attention on the category with the lowest F1-score. Across all feature sets, the lowest score belongs to the Fridges category. The Word count dataset slightly outperforms the others (F1-score = 0.91), which is why we propose using it for further experimentation with this algorithm.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = RandomForestClassifier()
    train_and_evaluate(model, "Random Forest")


Random Forest - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       766
 Digital Cameras       1.00      0.99      0.99       538
     Dishwashers       0.93      0.93      0.93       681
        Freezers       0.94      0.93      0.94       440
 Fridge Freezers       0.95      0.92      0.94      1094
         Fridges       0.87      0.91      0.89       712
      Microwaves       0.89      0.96      0.93       466
   Mobile Phones       0.98      0.98      0.98       812
             TVs       0.99      0.99      0.99       708
Washing Machines       0.96      0.94      0.95       803

        accuracy                           0.95      7020
       macro avg       0.95      0.95      0.95      7020
    weighted avg       0.95      0.95      0.95      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               763                0            0         0   
Digital C

6.5. Support Vector Machine

Please find the metric results below. Across all feature sets, the lowest F1-score belongs to the Fridges category. The Special character count dataset slightly outperforms the others (F1-score = 0.94), which is why we propose using it for further experimentation with this algorithm.

In [21]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from scipy.sparse import hstack
from sklearn.metrics import confusion_matrix

for name, features in feature_sets.items():
    X_train_final, X_test_final = get_feature_sets(features)
    model = LinearSVC()
    train_and_evaluate(model, "Support Vector Machine")


Support Vector Machine - Baseline
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       766
 Digital Cameras       1.00      1.00      1.00       538
     Dishwashers       0.94      0.98      0.96       681
        Freezers       0.99      0.96      0.97       440
 Fridge Freezers       0.96      0.95      0.96      1094
         Fridges       0.94      0.92      0.93       712
      Microwaves       0.99      0.97      0.98       466
   Mobile Phones       0.99      0.99      0.99       812
             TVs       0.97      1.00      0.98       708
Washing Machines       0.95      0.96      0.95       803

        accuracy                           0.97      7020
       macro avg       0.97      0.97      0.97      7020
    weighted avg       0.97      0.97      0.97      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               765                0            0         0   


6.6. Overall Comparison of Algorithms and Datasets

The highest F1-score, precision, and accuracy are achieved by the Support Vector Machine model trained on the dataset that combines the basic product title feature with the number of special characters in the title.

Therefore, this combination will be selected for further experimentation using standard TF-IDF vectorization with unigrams only and with unigrams, bigrams, and trigrams.

7. Experimenting with Two Additional TF-IDF Vectorization Types

Model: Support Vector Machine (SVM)  
Dataset: Product title + number of special characters in the title

7.1. Unigram

Please review the metrics below. The F1-score, precision, and other evaluation metrics are slightly lower than those obtained using unigrams and bigrams.

In [26]:
from sklearn.metrics import accuracy_score

# Add special_count to TF-IDF features

special_index = numeric_features.index("special_count")

X_train_special_unigram = hstack([
    X_train_unigram,
    X_train_num[:, special_index].reshape(-1, 1)
])

X_test_special_unigram = hstack([
    X_test_unigram,
    X_test_num[:, special_index].reshape(-1, 1)
])


# Train SVM
model = LinearSVC()
model.fit(X_train_special_unigram, y_train)
y_pred = model.predict(X_test_special_unigram)

# Evaluation
print(f"\n{'='*60}")
print("SVM - Special Count - Unigram")
print(f"{'='*60}")
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
))


SVM - Special Count - Unigram
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       766
 Digital Cameras       1.00      1.00      1.00       538
     Dishwashers       0.95      0.97      0.96       681
        Freezers       0.98      0.97      0.97       440
 Fridge Freezers       0.97      0.96      0.96      1094
         Fridges       0.94      0.93      0.93       712
      Microwaves       0.99      0.97      0.98       466
   Mobile Phones       0.99      0.99      0.99       812
             TVs       0.97      1.00      0.98       708
Washing Machines       0.95      0.96      0.96       803

        accuracy                           0.97      7020
       macro avg       0.97      0.97      0.97      7020
    weighted avg       0.97      0.97      0.97      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               764                0            0         0   
Digi

7.2. Trigram

Please review the metrics below. The F1-score, precision, and other evaluation metrics are slightly lower than those obtained using unigrams and bigrams.

In [25]:
# Add special_count to Trigram TF-IDF features

special_index = numeric_features.index("special_count")

X_train_special_trigram = hstack([
    X_train_trigram,
    X_train_num[:, special_index].reshape(-1, 1)
])

X_test_special_trigram = hstack([
    X_test_trigram,
    X_test_num[:, special_index].reshape(-1, 1)
])

# Train SVM
model = LinearSVC()
model.fit(X_train_special_trigram, y_train)
y_pred = model.predict(X_test_special_trigram)

# Evaluation
print(f"\n{'='*60}")
print("SVM - Special Count - Trigram")
print(f"{'='*60}")
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
print("Confusion Matrix:")
print(pd.DataFrame(
    cm,
    index=model.classes_,
    columns=model.classes_
))


SVM - Special Count - Trigram
                  precision    recall  f1-score   support

            CPUs       1.00      1.00      1.00       766
 Digital Cameras       0.99      1.00      1.00       538
     Dishwashers       0.94      0.98      0.96       681
        Freezers       0.98      0.96      0.97       440
 Fridge Freezers       0.96      0.95      0.95      1094
         Fridges       0.94      0.92      0.93       712
      Microwaves       0.99      0.97      0.98       466
   Mobile Phones       0.99      0.99      0.99       812
             TVs       0.97      1.00      0.98       708
Washing Machines       0.95      0.96      0.95       803

        accuracy                           0.97      7020
       macro avg       0.97      0.97      0.97      7020
    weighted avg       0.97      0.97      0.97      7020

Confusion Matrix:
                  CPUs  Digital Cameras  Dishwashers  Freezers  \
CPUs               765                0            0         0   
Digi

8. Final Choice

For further testing, we select the SVM model combined with the dataset containing the special character count and unigram + bigram TF-IDF vectorization.

Compared with the baseline dataset and unigram-only vectorization, these changes result in an improvement of only 0.01 in the F1-score. Therefore, the simpler SVM + baseline dataset + unigram vectorization configuration remains a reasonable alternative for further testing.